# Add a New Style to the Gallery

Run cells **1 → 5** in order.  
Cell 3 opens a file-picker dialog; cells 4–5 are the interactive inputs; cell 6 writes everything to disk.

---

### Model file formats

| File | What it is | Needed at… |
|---|---|---|
| `.pth` | PyTorch state-dict (pickle). Raw weight tensors. Requires PyTorch to load. | Training / re-export only |
| `.onnx` | Open Neural Network Exchange. Self-contained inference graph **+ embedded weights**. Framework-agnostic — runs on ONNX Runtime without PyTorch. | Runtime (app uses this) |
| `.onnx.data` | External weight blob produced automatically when the model exceeds ~2 GB (protobuf limit). The `.onnx` file stores the graph + references; the `.data` file stores the actual bytes. **Both files must stay in the same folder.** | Runtime (alongside `.onnx`) |

**Do they need to be converted?**  
No manual conversion is needed between `.onnx` and `.onnx.data` — they are produced together by `torch.onnx.export()`.  
The Kaggle training notebook already exports both.  
The `.pth` is kept as a backup for future fine-tuning or re-export only.

In [ ]:
import json
import pathlib
import shutil
import sys

REPO_ROOT     = pathlib.Path("..").resolve()
STYLES_DIR    = REPO_ROOT / "styles"
CATALOG_PATH  = STYLES_DIR / "catalog.json"
CONTENT_IMAGE = REPO_ROOT / "sample_images" / "arch.png"   # used for preview render

assert CATALOG_PATH.exists(), f"Catalog not found: {CATALOG_PATH}"
assert CONTENT_IMAGE.exists(), f"Content image not found: {CONTENT_IMAGE}"

with open(CATALOG_PATH, encoding="utf-8") as f:
    catalog: dict = json.load(f)

existing_ids: set[str] = {s["id"] for s in catalog["styles"]}
print(f"Repository   : {REPO_ROOT}")
print(f"Existing styles: {sorted(existing_ids)}")

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

onnx_str = filedialog.askopenfilename(
    title="Select the .onnx model file",
    filetypes=[("ONNX model", "*.onnx")],
)
root.destroy()

if not onnx_str:
    raise SystemExit("No file selected — re-run this cell to try again.")

onnx_path = pathlib.Path(onnx_str)
pth_path  = onnx_path.with_suffix(".pth")
data_path = pathlib.Path(str(onnx_path) + ".data")  # present only for large models

print(f".onnx : {onnx_path}")
print(f".pth  : {pth_path}")

In [ ]:
assert onnx_path.exists(), f"ONNX file not found: {onnx_path}"

has_pth  = pth_path.exists()
has_data = data_path.exists()

print(f"  .onnx      OK  ({onnx_path.stat().st_size / 1e6:.1f} MB)")
if has_pth:
    print(f"  .pth       OK  ({pth_path.stat().st_size  / 1e6:.1f} MB)")
else:
    print("  .pth       -- not present (optional; only needed for NST models)")
if has_data:
    print(f"  .onnx.data OK  ({data_path.stat().st_size / 1e6:.1f} MB)  -- will be copied too")
else:
    print("  .onnx.data -- not present (weights embedded in .onnx)")


In [ ]:
import ipywidgets as widgets
from IPython.display import display

name_w = widgets.Text(
    placeholder="e.g. Anime Hayao",
    description="Style name:",
    layout=widgets.Layout(width="380px"),
)
desc_w = widgets.Text(
    placeholder="e.g. Anime illustration style by AnimeGANv3",
    description="Description:",
    layout=widgets.Layout(width="520px"),
    style={"description_width": "90px"},
)
author_w = widgets.Text(
    value="PeterWazinski",
    description="Author:",
    layout=widgets.Layout(width="380px"),
)
layout_w = widgets.Dropdown(
    options=[("NST TransformerNet  (default)", "nchw"), ("AnimeGANv3 / TF NHWC tanh", "nhwc_tanh")],
    value="nchw",
    description="Layout:",
    layout=widgets.Layout(width="520px"),
    style={"description_width": "90px"},
)
status_out = widgets.Output()

def _check(_: object = None) -> None:
    with status_out:
        status_out.clear_output(wait=True)
        raw = name_w.value.strip()
        if not raw:
            return
        sid = raw.lower().replace(" ", "_")
        if sid in existing_ids:
            print(f"Warning: ID '{sid}' already exists in catalog -- choose a different name.")
        else:
            print(f"OK  ID will be: '{sid}'  tensor_layout='{layout_w.value}'")

name_w.observe(_check, names="value")
layout_w.observe(_check, names="value")
display(widgets.VBox([name_w, desc_w, author_w, layout_w, status_out]))


In [ ]:
STYLE_NAME    = name_w.value.strip()
STYLE_DESC    = desc_w.value.strip()
STYLE_AUTHOR  = author_w.value.strip() or "PeterWazinski"
TENSOR_LAYOUT = layout_w.value

assert STYLE_NAME, "Style name is empty -- fill in Cell 5 first."
STYLE_ID = STYLE_NAME.lower().replace(" ", "_")
assert STYLE_ID not in existing_ids, f"'{STYLE_ID}' already exists in the catalog."

dest_dir = STYLES_DIR / STYLE_ID
dest_dir.mkdir(parents=True, exist_ok=True)

has_pth  = pth_path.exists()
has_data = data_path.exists()

# -- Copy model files
shutil.copy2(onnx_path, dest_dir / "model.onnx")
if has_pth:
    shutil.copy2(pth_path, dest_dir / "model.pth")
    print("Copied model.pth")
if has_data:
    shutil.copy2(data_path, dest_dir / "model.onnx.data")
    print("Copied model.onnx.data")

# -- Generate preview
import importlib
sys.path.insert(0, str(REPO_ROOT))
import src.trainer.preview as _preview_mod
importlib.reload(_preview_mod)
from src.trainer.preview import generate_preview

preview_path = dest_dir / "preview.jpg"
generate_preview(
    onnx_path=dest_dir / "model.onnx",
    preview_path=preview_path,
    content_image=CONTENT_IMAGE,
    size=256,
    tensor_layout=TENSOR_LAYOUT,
)
print(f"Preview generated: {preview_path}")

# -- Update catalog.json
catalog["styles"].append({
    "id":            STYLE_ID,
    "name":          STYLE_NAME,
    "description":   STYLE_DESC,
    "author":        STYLE_AUTHOR,
    "model_path":    f"styles/{STYLE_ID}/model.onnx",
    "preview_path":  f"styles/{STYLE_ID}/preview.jpg",
    "is_builtin":    False,
    "tensor_layout": TENSOR_LAYOUT,
})
with open(CATALOG_PATH, "w", encoding="utf-8") as f:
    json.dump(catalog, f, indent=2)

print(f"\nOK  '{STYLE_NAME}'  (id='{STYLE_ID}', layout='{TENSOR_LAYOUT}')  added to gallery.")
print("Files:")
for p in sorted(dest_dir.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")
print("\nNext: git add -A ; git commit -m 'feat: add <name> style'")